# `aidp-iceberg` live test — Hadoop catalog on `oci://`
Registers an Iceberg catalog backed by an OCI bucket, creates a database + partitioned table, inserts rows, queries snapshots, time-travels.


In [ ]:
import sys, os
# Adjust this if you've uploaded the plugin scripts/ dir elsewhere.
sys.path.insert(0, '/Workspace/Shared/oracle_ai_data_platform_connectors/scripts')


In [ ]:
OCI_NAMESPACE = os.environ['OCI_NAMESPACE']
BUCKET_NAME   = os.environ['ICEBERG_BUCKET']
WAREHOUSE     = f'oci://{BUCKET_NAME}@{OCI_NAMESPACE}/iceberg-warehouse'
CATALOG       = os.environ.get('ICEBERG_CATALOG', 'oci_catalog')
DB            = os.environ.get('ICEBERG_DB', 'demo_db')
TABLE         = os.environ.get('ICEBERG_TABLE', 'employees')
FQN           = f'{CATALOG}.{DB}.{TABLE}'
spark.conf.set(f'spark.sql.catalog.{CATALOG}',           'org.apache.iceberg.spark.SparkCatalog')
spark.conf.set(f'spark.sql.catalog.{CATALOG}.type',      'hadoop')
spark.conf.set(f'spark.sql.catalog.{CATALOG}.warehouse', WAREHOUSE)
print('warehouse:', WAREHOUSE)


In [ ]:
spark.sql(f'CREATE DATABASE IF NOT EXISTS {CATALOG}.{DB}')
spark.sql(f'DROP TABLE IF EXISTS {FQN}')
spark.sql(f'''CREATE TABLE {FQN} (id INT, name STRING, dept STRING) USING iceberg PARTITIONED BY (dept)''')
data = [(1,'Alice','eng'), (2,'Bob','eng'), (3,'Carol','sales')]
spark.createDataFrame(data, ['id','name','dept']).writeTo(FQN).append()


In [ ]:
df = spark.sql(f'SELECT * FROM {FQN} ORDER BY id')
df.show()


In [ ]:
# Live-test driver parses this final cell's stdout for the JSON summary.
import json, time
summary = {
    'connector': 'aidp-iceberg',
    'auth': 'hadoop-catalog-oci',
    'rows': int(df.count()),
    'schema': sorted([f.name for f in df.schema.fields]),
    'timestamp_utc': int(time.time()),
}
print('AIDP_LIVE_TEST_RESULT_BEGIN')
print(json.dumps(summary, indent=2))
print('AIDP_LIVE_TEST_RESULT_END')
